# 📖 Notebook 2: Real-Time Driver Tracking

Uber has ~5 million active drivers. Each one sends their GPS location every 5 seconds. That's about **2 million updates per second** — far more than any traditional database can handle.

This notebook explores how to absorb this firehose of location data using Redis, keep the geo index fresh, and clean up stale drivers who go offline.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why a traditional database can't handle millions of writes per second
- How Redis GEOADD overwrites previous locations (upsert behavior)
- How to detect and remove stale drivers
- How adaptive update intervals reduce server load

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/uber
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import math

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "uber_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

# Every simulation below jitters positions and timestamps. Seed the RNG so a
# re-run produces the same numbers and the assertions mean something.
random.seed(42)

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Problem: 2 Million Writes per Second

Let's do the math on why a regular database can't handle this:

```
5,000,000 active drivers
× 1 update every 5 seconds
= 1,000,000 writes/sec (average)
≈ 2,000,000 writes/sec (peak)
```

PostgreSQL can handle ~10,000–50,000 writes/sec on good hardware.  
DynamoDB at 2M writes/sec would cost ~$200,000/day.

**Redis handles 100,000+ writes/sec on a single node** and can be sharded for even more. Since location data is ephemeral (only the latest position matters), Redis's in-memory nature is a perfect fit.

In [ ]:
# Let's measure write throughput: PostgreSQL vs Redis

# Simulate 1000 location updates to PostgreSQL
conn = get_db()
conn.autocommit = True
cur = conn.cursor()

# This benchmark overwrites every driver's real position with random jitter.
# Notebooks 1, 3 and 4 all read driver_locations and reason about *where* the
# drivers actually are, so snapshot the seeded positions and put them back when
# we're done. A benchmark that silently corrupts the fixture is how you end up
# debugging notebook 3 for a bug that lives here.
cur.execute("""
    SELECT driver_id,
           ST_X(location::geometry) AS lng,
           ST_Y(location::geometry) AS lat
    FROM driver_locations ORDER BY driver_id;
""")
seeded_locations = cur.fetchall()

start = time.time()
for i in range(1000):
    # Use driver_id 1-10, cycling through them
    driver_id = (i % 10) + 1
    lng = -122.4194 + random.uniform(-0.01, 0.01)
    lat = 37.7749 + random.uniform(-0.01, 0.01)
    cur.execute("""
        INSERT INTO driver_locations (driver_id, location, updated_at)
        VALUES (%s, ST_SetSRID(ST_MakePoint(%s, %s), 4326), NOW())
        ON CONFLICT (driver_id) DO UPDATE 
        SET location = ST_SetSRID(ST_MakePoint(%s, %s), 4326),
            updated_at = NOW();
    """, (driver_id, lng, lat, lng, lat))
pg_elapsed = time.time() - start

# Put the fixture back exactly as we found it.
for did, lng, lat in seeded_locations:
    cur.execute("""
        UPDATE driver_locations
        SET location = ST_SetSRID(ST_MakePoint(%s, %s), 4326), updated_at = NOW()
        WHERE driver_id = %s;
    """, (lng, lat, did))

cur.execute("""
    SELECT driver_id,
           ST_X(location::geometry) AS lng,
           ST_Y(location::geometry) AS lat
    FROM driver_locations ORDER BY driver_id;
""")
after = cur.fetchall()
conn.close()

assert len(after) == len(seeded_locations), (
    f"driver_locations row count changed: {len(seeded_locations)} -> {len(after)}"
)
for (did, lng, lat), (did2, lng2, lat2) in zip(seeded_locations, after):
    assert did == did2 and abs(lng - lng2) < 1e-9 and abs(lat - lat2) < 1e-9, (
        f"driver {did} was not restored: ({lat}, {lng}) -> ({lat2}, {lng2})"
    )

# Simulate 1000 location updates to Redis
r = get_redis()
start = time.time()
for i in range(1000):
    driver_id = (i % 10) + 1
    lng = -122.4194 + random.uniform(-0.01, 0.01)
    lat = 37.7749 + random.uniform(-0.01, 0.01)
    r.geoadd("drivers:locations", (lng, lat, f"driver:{driver_id}"))
redis_elapsed = time.time() - start

pg_rate = 1000 / pg_elapsed
redis_rate = 1000 / redis_elapsed

print(f"📊 Write Throughput (1000 location updates):")
print(f"  PostgreSQL: {pg_elapsed:.3f}s → {pg_rate:,.0f} writes/sec")
print(f"  Redis:      {redis_elapsed:.3f}s → {redis_rate:,.0f} writes/sec")
print()
print(f"🚀 Redis is {redis_rate / pg_rate:.1f}× faster for location updates!")
print()
print("💡 And this is with just 10 drivers. At millions of drivers,")
print("   PostgreSQL would need expensive sharding while Redis handles it easily.")

# The entire "Redis as a write buffer" argument rests on this gap. If PostgreSQL
# ever wins here, the rest of the notebook is telling you a story, not a fact.
assert redis_rate > pg_rate, (
    f"expected Redis to absorb location writes faster than PostgreSQL, got "
    f"redis={redis_rate:,.0f}/s vs postgres={pg_rate:,.0f}/s"
)

## 📍 Simulating Real-Time Driver Movement

Let's simulate drivers moving around San Francisco, sending location updates every few seconds. Each GEOADD **overwrites** the previous location for that driver — so the geo set always has the latest position.

This is a crucial property: Redis Geo gives us "last-write-wins" upsert behavior for free.

In [ ]:
r = get_redis()
r.delete("drivers:locations", "drivers:last_seen", "drivers:available")

# Starting positions for 10 drivers in SF
drivers = {
    1: {"lng": -122.4194, "lat": 37.7749, "name": "Alex"},
    2: {"lng": -122.4089, "lat": 37.7837, "name": "Bella"},
    3: {"lng": -122.3930, "lat": 37.7956, "name": "Carlos"},
    4: {"lng": -122.4378, "lat": 37.7594, "name": "Diana"},
    5: {"lng": -122.4580, "lat": 37.7694, "name": "Ethan"},
    6: {"lng": -122.4343, "lat": 37.8024, "name": "Fiona"},
    7: {"lng": -122.4862, "lat": 37.7568, "name": "George"},
    8: {"lng": -122.4103, "lat": 37.7627, "name": "Hannah"},
    9: {"lng": -122.3895, "lat": 37.7867, "name": "Ivan"},
   10: {"lng": -122.4474, "lat": 37.7229, "name": "Julia"},
}

# All drivers start as available
for did in drivers:
    r.sadd("drivers:available", f"driver:{did}")

def simulate_movement(driver_id, steps=5):
    """Simulate a driver moving, sending location updates each step."""
    d = drivers[driver_id]
    lng, lat = d["lng"], d["lat"]
    
    print(f"  🚗 Driver {driver_id} ({d['name']}) moving...")
    for step in range(steps):
        # Small random movement (~100m per step)
        lng += random.uniform(-0.001, 0.001)
        lat += random.uniform(-0.001, 0.001)
        
        # Update location in Redis (overwrites previous position)
        r.geoadd("drivers:locations", (lng, lat, f"driver:{driver_id}"))
        
        # Track when we last heard from this driver
        r.zadd("drivers:last_seen", {f"driver:{driver_id}": time.time()})
        
        print(f"     Step {step+1}: ({lat:.5f}, {lng:.5f})")
    
    # Update stored position
    drivers[driver_id]["lng"] = lng
    drivers[driver_id]["lat"] = lat

# Simulate 3 drivers moving
simulate_movement(1, steps=3)
simulate_movement(2, steps=3)
simulate_movement(3, steps=3)

print()
print("💡 Each GEOADD overwrites the old position — the geo set always shows")
print("   the latest location. No duplicates, no cleanup needed.")

## 🧹 Staleness Detection: Removing Offline Drivers

What happens when a driver closes the app, loses signal, or their phone dies? They stop sending updates, but their last location is still in the geo set. A rider might get matched with a "ghost" driver who isn't actually there.

**Solution**: Track the timestamp of each driver's last update. Periodically remove drivers whose last update is older than a threshold (e.g., 30 seconds).

We use a Redis **sorted set** (`drivers:last_seen`) where:
- Member = driver key
- Score = Unix timestamp of last update

In [ ]:
# Let's set up a scenario where some drivers go "stale"

r = get_redis()
now = time.time()

# Simulate: drivers 1-5 sent updates recently (within 10 seconds)
for did in range(1, 6):
    d = drivers[did]
    r.geoadd("drivers:locations", (d["lng"], d["lat"], f"driver:{did}"))
    r.zadd("drivers:last_seen", {f"driver:{did}": now - random.uniform(0, 10)})

# Simulate: drivers 6-8 sent updates 45+ seconds ago (stale!)
for did in range(6, 9):
    d = drivers[did]
    r.geoadd("drivers:locations", (d["lng"], d["lat"], f"driver:{did}"))
    r.zadd("drivers:last_seen", {f"driver:{did}": now - random.uniform(45, 90)})

# Simulate: drivers 9-10 haven't sent any update (missing from last_seen)
for did in range(9, 11):
    d = drivers[did]
    r.geoadd("drivers:locations", (d["lng"], d["lat"], f"driver:{did}"))

print("📊 Driver last-seen timestamps:")
print(f"{'Driver':<14} {'Last Seen':>12} {'Status':>10}")
print("-" * 40)

threshold_sec = 30
all_members = r.zrangebyscore("drivers:last_seen", "-inf", "+inf", withscores=True)
for member, score in sorted(all_members, key=lambda x: x[0]):
    age = now - score
    status = "✅ fresh" if age < threshold_sec else "⚠️ STALE"
    print(f"{member:<14} {age:>9.1f}s ago {status:>10}")

# Show drivers not in last_seen at all
for did in [9, 10]:
    print(f"driver:{did:<7} {'never':>12} {'❌ UNKNOWN':>10}")

In [ ]:
def cleanup_stale_drivers(threshold_seconds=30):
    """
    Remove drivers who haven't sent an update in the last `threshold_seconds`.
    This runs periodically (e.g., every 10 seconds) as a background task.

    There are TWO kinds of ghost to sweep, and the second one is easy to miss:

      1. Drivers whose `last_seen` score is older than the cutoff.
      2. Drivers sitting in the geo set with NO `last_seen` entry at all.

    Kind 2 happens whenever a driver lands in the geo set through some path that
    doesn't stamp a timestamp — a backfill, a replay, a bug. A cleanup that only
    walks `drivers:last_seen` will never even look at them, so they stay
    matchable forever: permanently fresh-looking, permanently not there.
    """
    r = get_redis()
    cutoff = time.time() - threshold_seconds

    # Kind 1 — last update is older than the cutoff
    stale = set(r.zrangebyscore("drivers:last_seen", "-inf", cutoff))

    # Kind 2 — in the geo set but never checked in. Treat as infinitely stale.
    in_geo = set(r.zrange("drivers:locations", 0, -1))
    tracked = set(r.zrange("drivers:last_seen", 0, -1))
    stale |= in_geo - tracked

    for driver_key in sorted(stale):
        # Remove from geo set (no longer searchable)
        r.zrem("drivers:locations", driver_key)
        # Remove from available set
        r.srem("drivers:available", driver_key)
        # Remove from last_seen tracking
        r.zrem("drivers:last_seen", driver_key)

    return sorted(stale)


# Snapshot the "never checked in" ghosts before we sweep, so we can prove they went.
before_geo = set(r.zrange("drivers:locations", 0, -1))
never_checked_in = before_geo - set(r.zrange("drivers:last_seen", 0, -1))
print(f"👻 In the geo set with no last_seen entry at all: {sorted(never_checked_in)}")
print()

# Run cleanup
removed = cleanup_stale_drivers(threshold_seconds=30)
print(f"🧹 Removed {len(removed)} stale drivers: {removed}")
print()

# Verify: how many drivers remain in the geo set?
remaining_geo = set(r.zrange("drivers:locations", 0, -1))
available = r.scard("drivers:available")
print(f"📊 After cleanup:")
print(f"   Drivers in geo set:       {len(remaining_geo)}  {sorted(remaining_geo)}")
print(f"   Drivers marked available: {available}")
print()
print("💡 Stale drivers are gone — riders won't be matched with ghost drivers!")
print("   In production, this cleanup runs every 10 seconds as a cron/background job.")

# The scenario above deliberately plants both kinds of ghost. Both must be gone.
assert never_checked_in, (
    "the previous cell was supposed to leave drivers 9 and 10 in the geo set with "
    "no last_seen entry — without them this cell can't demonstrate anything"
)
survivors = remaining_geo & never_checked_in
assert not survivors, (
    f"ghost drivers survived cleanup: {sorted(survivors)}. A cleanup that only "
    "walks drivers:last_seen never sees a driver that never checked in."
)
assert remaining_geo == {f"driver:{d}" for d in range(1, 6)}, (
    f"expected only the five fresh drivers to remain, got {sorted(remaining_geo)}"
)
assert available == 5, f"expected 5 drivers still marked available, got {available}"

## 📱 Adaptive Update Intervals

Sending GPS every 5 seconds from millions of drivers is expensive. Can we be smarter?

**Adaptive intervals** adjust the update frequency based on context:

| Situation | Interval | Why |
|-----------|----------|-----|
| Driver is parked/stationary | 30 seconds | Position isn't changing |
| Driver moving on highway | 10 seconds | Moving fast but straight |
| Driver in city traffic (< 30 km/h) | 5 seconds | Direction changes frequently |
| Driver has active ride | 5 seconds | Need accurate ETA for rider |
| Driver near pending pickup | 2 seconds | Rider is watching the map closely |

The table is exactly what `calculate_update_interval` below returns — keep the two in
sync, because a table that quietly disagrees with the code is how a lab starts lying.

This logic runs **on the driver's phone**, not on the server. The phone has GPS, accelerometer, and gyroscope data to make these decisions locally.

In [ ]:
def calculate_update_interval(speed_kmh, has_active_ride, near_pickup):
    """
    Determine how often the driver's app should send location updates.
    This logic runs on the client device.
    
    Args:
        speed_kmh: Current speed from GPS
        has_active_ride: Whether the driver has an active ride
        near_pickup: Whether the driver is near the pickup point
    
    Returns:
        Interval in seconds between location updates
    """
    if near_pickup:
        return 2  # rider is watching the map closely
    
    if has_active_ride:
        return 5  # need ETA accuracy
    
    if speed_kmh < 2:
        return 30  # basically parked -- position isn't changing
    if speed_kmh < 30:
        return 5   # city driving -- frequent turns
    return 10      # highway -- fast, but in straight, predictable lines


# Show the effect of adaptive intervals
scenarios = [
    (0, False, False, "Parked, no ride"),
    (25, False, False, "City driving, no ride"),
    (60, False, False, "Highway, no ride"),
    (25, True, False, "City driving, active ride"),
    (10, True, True, "Approaching pickup"),
]

print("📱 Adaptive Update Intervals:")
print(f"{'Scenario':<30} {'Speed':>8} {'Interval':>10} {'Updates/min':>12}")
print("-" * 65)
for speed, ride, pickup, label in scenarios:
    interval = calculate_update_interval(speed, ride, pickup)
    per_min = 60 / interval
    print(f"{label:<30} {speed:>5} km/h {interval:>7}s {per_min:>10.1f}")

print()

# Calculate savings
fixed_rate = 5_000_000 / 5  # 5M drivers at 5 sec interval
# Assume: 30% parked, 40% city, 20% highway, 10% active ride
adaptive_rate = (
    5_000_000 * 0.30 / 30 +   # parked
    5_000_000 * 0.40 / 5 +    # city
    5_000_000 * 0.20 / 10 +   # highway
    5_000_000 * 0.10 / 5      # active ride
)

reduction_pct = (1 - adaptive_rate / fixed_rate) * 100

print(f"📊 Impact at scale (5M drivers):")
print(f"  Fixed 5s interval:    {fixed_rate:>12,.0f} updates/sec")
print(f"  Adaptive intervals:   {adaptive_rate:>12,.0f} updates/sec")
print(f"  Reduction:            {reduction_pct:>11.0f}%")
print()
print(f"💡 Adaptive intervals cut server load by {reduction_pct:.0f}% for THIS traffic mix.")
print("   Look at where the saving comes from: the parked 30% drop from 12 updates/min")
print("   to 2, and that is nearly the whole story. City driving stays at 5s, so it")
print("   contributes exactly nothing. Shift the mix toward city driving and the saving")
print("   shrinks — which is why you measure this per-market instead of quoting a")
print("   headline number.")

assert adaptive_rate < fixed_rate, (
    f"adaptive intervals must not increase load: {adaptive_rate:,.0f} vs {fixed_rate:,.0f}"
)
assert reduction_pct > 25, (
    f"expected adaptive intervals to cut load by more than 25% for this mix, "
    f"got {reduction_pct:.0f}%"
)

## 🔄 Putting It All Together: The Location Update Pipeline

Here's the complete flow for handling driver location updates:

```
Driver's Phone                    Server
─────────────                    ──────
GPS sensor fires          
    │                     
    ▼                     
Calculate adaptive        
interval (on device)      
    │                     
    ▼                     
Time to send? ─── No ──→ Wait
    │                     
   Yes                    
    │                     
    ▼                     
POST /drivers/location ──→ Location Service
                               │
                               ├──→ Redis GEOADD (update position)
                               │
                               └──→ Redis ZADD last_seen (update timestamp)


Background Cleanup Job (every 10s)
─────────────────────────────────
Find drivers with last_seen > 30s ago
    │
    ▼
Remove from geo set + available set
```

In [ ]:
# Complete simulation: multiple drivers updating, a cleanup cycle, then a search

r = get_redis()
r.delete("drivers:locations", "drivers:last_seen", "drivers:available")

now = time.time()

print("=" * 60)
print("Step 1: Drivers send location updates")
print("=" * 60)

# 8 active drivers send updates
for did in range(1, 9):
    d = drivers[did]
    lng = d["lng"] + random.uniform(-0.002, 0.002)
    lat = d["lat"] + random.uniform(-0.002, 0.002)
    
    r.geoadd("drivers:locations", (lng, lat, f"driver:{did}"))
    r.zadd("drivers:last_seen", {f"driver:{did}": now})
    r.sadd("drivers:available", f"driver:{did}")
    print(f"  ✅ driver:{did} updated at ({lat:.4f}, {lng:.4f})")

# 2 drivers are "stale" (last update 60 seconds ago)
for did in [9, 10]:
    d = drivers[did]
    r.geoadd("drivers:locations", (d["lng"], d["lat"], f"driver:{did}"))
    r.zadd("drivers:last_seen", {f"driver:{did}": now - 60})  # 60 sec stale
    r.sadd("drivers:available", f"driver:{did}")
    print(f"  ⚠️  driver:{did} last seen 60s ago (stale!)")

print(f"\n  Total in geo set: {r.zcard('drivers:locations')}")
print(f"  Total available:  {r.scard('drivers:available')}")

print()
print("=" * 60)
print("Step 2: Cleanup job removes stale drivers")
print("=" * 60)
removed = cleanup_stale_drivers(threshold_seconds=30)
print(f"  Removed: {removed}")
print(f"  Remaining in geo set: {r.zcard('drivers:locations')}")
print(f"  Remaining available:  {r.scard('drivers:available')}")

print()
print("=" * 60)
print("Step 3: Rider searches for nearby drivers")
print("=" * 60)
nearby = r.geosearch(
    name="drivers:locations",
    longitude=-122.4194, latitude=37.7749,
    radius=5, unit="km",
    withdist=True, sort="ASC", count=5
)
available = r.smembers("drivers:available")
matches = [(m, d) for m, d in nearby if m in available]

for member, dist in matches:
    print(f"  🚗 {member}: {float(dist):.2f} km away")

print()
print("✅ Only fresh, available drivers appear in search results!")

# Pin the lesson down. The pipeline is only correct if the two stale drivers are
# gone from the geo set AND absent from the rider's search results.
ghosts = {"driver:9", "driver:10"}
result_members = {m for m, _ in matches}
assert set(removed) == ghosts, (
    f"the cleanup job should have swept exactly {sorted(ghosts)}, got {removed}"
)
assert not (result_members & ghosts), (
    f"a stale driver is still matchable: {sorted(result_members & ghosts)}"
)
assert result_members <= available, (
    f"search returned drivers not in the available set: {sorted(result_members - available)}"
)
assert r.zcard("drivers:locations") == 8, (
    f"expected 8 fresh drivers left in the geo set, got {r.zcard('drivers:locations')}"
)

## 🧹 Cleanup

In [ ]:
r = get_redis()
r.delete("drivers:locations", "drivers:last_seen", "drivers:available")
print("🧹 Cleaned up Redis keys")

## 📚 Summary

### Key Takeaways

1. **Traditional databases can't handle 2M writes/sec** — Redis can because it's in-memory
2. **GEOADD is an upsert** — each update overwrites the previous location, keeping only the latest
3. **Staleness detection** is critical — and there are *two* kinds of ghost: a driver whose
   `last_seen` went stale, and a driver in the geo set that never had a `last_seen` at all.
   A cleanup that only walks `drivers:last_seen` leaves the second kind matchable forever
4. **Adaptive intervals** reduce server load by ~35% for the mix modelled here — the phone
   decides when to send based on speed and context. Almost all of that saving comes from
   parked drivers; the number moves with the fleet mix, so measure it, don't quote it
5. **Data durability is OK to sacrifice** — if Redis crashes, drivers re-send their position within 5 seconds

### How This Fits in a System Design Interview

When asked "how do you handle location updates at scale?":
- ❌ Bad: write every update to PostgreSQL → can't handle the write volume
- ✅ Good: batch writes to PostgreSQL → reduces volume but adds staleness
- ✅ Great: Redis Geo + adaptive intervals → real-time, scalable, ephemeral

### Next Up

In **Notebook 3**, we'll build **surge pricing** — how Uber adjusts prices based on supply and demand in each zone.